# Imports

In [1]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import sklearn
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm

from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd
from datasets import load_dataset
import json

ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
from scripts.download_leaderboard import MMLU_SUBSCENARIOS
# from utils_for_notebooks import read_per_model_info
from utils_for_notebooks import pad_predictions
from scripts.evaluate_mmlu import evaluate_mmlu
sys.path.pop(0);


CACHE_DIR = "./cache_dir"


/home/oh/arubinstein17/github/disco-public/envs/disco_env_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Functions

In [7]:
def acc_from_hf(target_model_name, scenario_name, timestamp):

    # scenario_name = "harness_hendrycksTest_abstract_algebra_5"
    # timestamp = '2024_01_16T04_06_55.134598'

    model_name_hf = openllmname_to_hf_model_id(target_model_name)
    model_df = read_per_model_info(
        model_id=model_name_hf,
        timestamp=timestamp,
        scenario=scenario_name,
        cache_dir=CACHE_DIR
    )
    acc, norm_acc, num_correct_acc, num_correct_norm_acc = compute_accuracy(model_df)
    return acc, norm_acc, num_correct_acc, num_correct_norm_acc, len(model_df)



def load_jsonl(filename):
    results = []
    with open(filename, 'r') as infile:
        for line in infile:
            results.append(json.loads(line))
    return results


# Convert DataFrame to target_outputs format
def convert_single_model_df(model_df, model_id, scenario, metric):
    """
    Convert a single model DataFrame from read_per_model_info to arrays.

    Parameters:
    - model_df: DataFrame from read_per_model_info
    - model_id: Model identifier (e.g., "meta-llama/Llama-2-13b-hf")
    - scenario: Scenario name (e.g., "harness_hellaswag_10")

    Returns:
    - predictions_2d: numpy array of shape (n_questions, n_choices)
    - correctness_1d: numpy array of shape (n_questions,)
    - n_questions: number of questions
    """
    n_questions = len(model_df)

    # Extract predictions - should be logits for each choice
    # The predictions column contains arrays of logits for each choice
    if "predictions" not in model_df.columns:
        raise ValueError("DataFrame missing 'predictions' column")

    predictions_list = []
    for idx, pred in enumerate(model_df["predictions"]):
        # Convert to numpy array if not already
        if isinstance(pred, (list, np.ndarray)):
            pred_array = np.array(pred)
        else:
            raise ValueError(f"Row {idx} predictions is not a list/array: {type(pred)}")
        predictions_list.append(pred_array)

    # Stack predictions: (n_questions, n_choices)
    predictions_2d = np.stack(predictions_list)

    # Extract correctness from "acc" column

    try:
        correctness_1d = np.array([
            a[metric] for a in model_df["metrics"]
        ], dtype=float)
    except:

        correctness_1d = np.array(model_df[metric].values, dtype=float)

    return predictions_2d, correctness_1d, n_questions


def convert_jsonl_results_to_arrays(jsonl_results, metric):
    """
    Convert JSONL results to predictions and correctness arrays.

    Parameters:
    - jsonl_results: List of dictionaries from load_jsonl

    Returns:
    - predictions_2d: numpy array of shape (n_questions, n_choices)
    - correctness_1d: numpy array of shape (n_questions,)
    - n_questions: number of questions
    """
    n_questions = len(jsonl_results)

    # Extract predictions from resps
    predictions_list = []
    correctness_list = []

    for result in jsonl_results:
        # Extract logits from resps
        # resps[i][0][0] gives the logit for choice i
        resps = result.get('resps', [])
        if not resps:
            raise ValueError(f"Result missing 'resps' field")

        # Extract logits for each choice
        choice_logits = []
        for resp in resps:
            if isinstance(resp, list) and len(resp) > 0:
                if isinstance(resp[0], list) and len(resp[0]) > 0:
                    logit_str = resp[0][0]
                    logit = float(logit_str)
                    choice_logits.append(logit)
                else:
                    raise ValueError(f"Unexpected resps structure: {resp}")
            else:
                raise ValueError(f"Unexpected resps structure: {resp}")

        predictions_list.append(choice_logits)

        # Extract correctness from 'acc' field (1.0 if correct, 0.0 if incorrect)
        acc = result.get(metric, 0.0)
        correctness_list.append(float(acc))

    # Stack predictions: (n_questions, n_choices)
    predictions_2d = np.array(predictions_list)

    # Extract correctness: (n_questions,)
    correctness_1d = np.array(correctness_list, dtype=float)

    return predictions_2d, correctness_1d, n_questions


def find_jsonl_file_in_directory(directory_path):
    """
    Find the JSONL file in a directory. Looks for files matching pattern:
    samples_*_prompts_*.jsonl

    Parameters:
    - directory_path: Path to directory containing the JSONL file

    Returns:
    - Path to JSONL file, or None if not found
    """
    if not os.path.isdir(directory_path):
        # If it's already a file, return it
        if os.path.isfile(directory_path) and directory_path.endswith('.jsonl'):
            return directory_path
        return None

    # Look for JSONL files matching the pattern
    for root, dirs, files in os.walk(directory_path):
        for file in files:
            if file.endswith('.jsonl') and 'samples' in file and 'prompts' in file:
                return os.path.join(root, file)

    # If not found, try to find any JSONL file
    for root, dirs, files in os.walk(directory_path):
        for file in files:
            if file.endswith('.jsonl'):
                return os.path.join(root, file)

    return None


def convert_model_paths_to_target_outputs(model_id_to_path_mapping, scenario, metric, output_path=None, pad_to_size=None):
    """
    Convert a mapping of model_id to local file paths to target_outputs format and save to pickle.

    Parameters:
    - model_id_to_path_mapping: Dictionary mapping model_id -> local file path (directory or JSONL file)
    - scenario: Scenario name (e.g., "harness_hellaswag_10")
    - metric: Metric name to use for correctness (e.g., "acc", "acc_norm")
    - output_path: Path to save target_outputs.pkl (if None, uses default)
    - pad_to_size: Optional size to pad predictions to on the last axis. If None, no padding is applied.

    Returns:
    - target_outputs: Dictionary with the expected structure
    """
    if output_path is None:
        # Extract scenario base name for default path
        if scenario.startswith("harness_"):
            scenario_base = scenario.replace("harness_", "")
            if "_" in scenario_base:
                scenario_base = scenario_base.split("_")[0]
        else:
            scenario_base = scenario
        output_path = f"/home/oh/arubinstein17/github/disco-public/data/model_outputs/{scenario_base}/target_outputs.pkl"

    # Check if file exists and load it to append
    existing_outputs = None
    if os.path.exists(output_path):
        print(f"Loading existing target_outputs from {output_path}")
        existing_outputs = load_pickle(output_path)
        # Extract existing model IDs
        existing_model_ids = set(existing_outputs["Models"].keys())
        print(f"Found {len(existing_model_ids)} existing models: {existing_model_ids}")
    else:
        print(f"Creating new target_outputs file at {output_path}")

    # Collect data for all models
    all_predictions = []
    all_correctness = []
    models_map = {}
    n_questions = None

    for model_idx, (model_id, path) in enumerate(model_id_to_path_mapping.items()):
        print(f"\nProcessing model {model_idx + 1}/{len(model_id_to_path_mapping)}: {model_id}")

        # Skip if model already exists
        if existing_outputs is not None and model_id in existing_outputs["Models"]:
            print(f"  Model {model_id} already exists, skipping...")
            continue

        # Find the JSONL file
        jsonl_path = find_jsonl_file_in_directory(path)
        if jsonl_path is None:
            print(f"  Error: Could not find JSONL file in {path}")
            continue

        print(f"  Found JSONL file: {jsonl_path}")

        # Load JSONL results
        try:
            jsonl_results = load_jsonl(jsonl_path)
        except Exception as e:
            print(f"  Error loading JSONL file {jsonl_path}: {e}")
            continue

        # Convert to arrays
        try:
            predictions_2d, correctness_1d, model_n_questions = convert_jsonl_results_to_arrays(jsonl_results, metric)
        except Exception as e:
            print(f"  Error converting JSONL results: {e}")
            continue

        # Pad predictions if pad_to_size is specified
        if pad_to_size is not None:
            n_choices = predictions_2d.shape[1]
            if n_choices < pad_to_size:
                # Pad each row using pad_predictions function
                padded_predictions_list = []
                for row in predictions_2d:
                    row_list = row.tolist()
                    padded_row = pad_predictions(row_list, max_num_answers=pad_to_size)
                    padded_predictions_list.append(padded_row)
                predictions_2d = np.array(padded_predictions_list)
                print(f"  Padded predictions from {n_choices} to {pad_to_size} choices")

        # Check consistency of number of questions
        if n_questions is None:
            n_questions = model_n_questions
        elif n_questions != model_n_questions:
            raise ValueError(
                f"Model {model_id} has {model_n_questions} questions, "
                f"but expected {n_questions}"
            )

        # Add to lists
        all_predictions.append(predictions_2d)
        all_correctness.append(correctness_1d)

        # Determine model index (append to existing or use new index)
        if existing_outputs is not None:
            # Find the next available index
            max_existing_idx = max(existing_outputs["Models"].values()) if existing_outputs["Models"] else -1
            model_index = max_existing_idx + 1
        else:
            model_index = model_idx

        models_map[model_id] = model_index
        print(f"  Added model {model_id} at index {model_index}")

    # Stack all models: (n_models, n_questions, n_choices) and (n_models, n_questions, 1)
    if existing_outputs is not None and len(all_predictions) == 0:
        print("No new models to add, keeping existing target_outputs")
        return existing_outputs

    if len(all_predictions) > 0:
        new_predictions = np.stack(all_predictions)  # (n_new_models, n_questions, n_choices)
        new_correctness = np.stack(all_correctness)[:, :, np.newaxis]  # (n_new_models, n_questions, 1)

        if existing_outputs is not None:
            # Concatenate with existing
            predictions = np.concatenate([existing_outputs["predictions"], new_predictions], axis=0)
            correctness = np.concatenate([existing_outputs["correctness"], new_correctness], axis=0)
            # Merge models map
            models_map = {**existing_outputs["Models"], **models_map}
            # Use existing datapoints and scenarios (should be the same)
            datapoints_map = existing_outputs["Datapoints"]
            scenarios_map = existing_outputs["Scenarios"]
        else:
            # Create new
            predictions = new_predictions
            correctness = new_correctness
            # Create Datapoints mapping: idx -> idx
            datapoints_map = {i: i for i in range(n_questions)}

            # Extract scenario base name
            if scenario.startswith("harness_"):
                scenario_base = scenario.replace("harness_", "")
                if "_" in scenario_base:
                    scenario_base = scenario_base.split("_")[0]
            else:
                scenario_base = scenario

            # Create Scenarios mapping: scenario_name -> list of all datapoint indices
            scenarios_map = {scenario_base: list(range(n_questions))}
    else:
        # Only existing models, no new ones
        predictions = existing_outputs["predictions"]
        correctness = existing_outputs["correctness"]
        models_map = existing_outputs["Models"]
        datapoints_map = existing_outputs["Datapoints"]
        scenarios_map = existing_outputs["Scenarios"]

    target_outputs = {
        "predictions": predictions,
        "correctness": correctness,
        "Models": models_map,
        "Datapoints": datapoints_map,
        "Scenarios": scenarios_map,
    }

    # Save to pickle file
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    dump_pickle(target_outputs, output_path)

    print(f"\nSaved target_outputs to {output_path}")
    print(f"Shape of predictions: {target_outputs['predictions'].shape}")
    print(f"Shape of correctness: {target_outputs['correctness'].shape}")
    print(f"Number of models: {len(target_outputs['Models'])}")
    print(f"Number of datapoints: {len(target_outputs['Datapoints'])}")
    print(f"Scenarios: {list(target_outputs['Scenarios'].keys())}")
    print(f"Model IDs: {list(target_outputs['Models'].keys())}")

    return target_outputs


def convert_model_ids_to_target_outputs(
    model_ids,
    scenario,
    metric,
    timestamp='latest',
    cache_dir=CACHE_DIR,
    output_path=None,
    pad_to_size=None
):
    """
    Convert a list of model_ids to target_outputs format and save to pickle.

    Parameters:
    - model_ids: List of model identifiers (e.g., ["meta-llama/Llama-2-13b-hf", ...])
    - scenario: Scenario name (e.g., "harness_hellaswag_10")
    - metric: Metric name to use for correctness (e.g., "acc", "acc_norm")
    - timestamp: Timestamp to use (default: 'latest')
    - cache_dir: Cache directory for datasets
    - output_path: Path to save target_outputs.pkl (if None, uses default)
    - pad_to_size: Optional size to pad predictions to on the last axis. If None, no padding is applied.

    Returns:
    - target_outputs: Dictionary with the expected structure
    """
    if output_path is None:
        # Extract scenario base name for default path
        if scenario.startswith("harness_"):
            scenario_base = scenario.replace("harness_", "")
            if "_" in scenario_base:
                scenario_base = scenario_base.split("_")[0]
        else:
            scenario_base = scenario
        output_path = f"/home/oh/arubinstein17/github/disco-public/data/model_outputs/{scenario_base}/target_outputs.pkl"

    # Check if file exists and load it to append
    existing_outputs = None
    if os.path.exists(output_path):
        print(f"Loading existing target_outputs from {output_path}")
        existing_outputs = load_pickle(output_path)
        # Extract existing model IDs
        existing_model_ids = set(existing_outputs["Models"].keys())
        print(f"Found {len(existing_model_ids)} existing models: {existing_model_ids}")
    else:
        print(f"Creating new target_outputs file at {output_path}")

    # Collect data for all models
    all_predictions = []
    all_correctness = []
    models_map = {}
    n_questions = None

    for model_idx, model_id in enumerate(model_ids):
        print(f"\nProcessing model {model_idx + 1}/{len(model_ids)}: {model_id}")

        # Skip if model already exists
        if existing_outputs is not None and model_id in existing_outputs["Models"]:
            print(f"  Model {model_id} already exists, skipping...")
            continue

        # Load model data
        try:
            model_df = read_per_model_info(
                model_id=model_id,
                timestamp=timestamp,
                scenario=scenario,
                cache_dir=cache_dir
            )
        except Exception as e:
            print(f"  Error loading model {model_id}: {e}")
            continue

        # Convert to arrays
        predictions_2d, correctness_1d, model_n_questions = convert_single_model_df(
            model_df, model_id, scenario, metric
        )

        # Pad predictions if pad_to_size is specified
        if pad_to_size is not None:
            n_choices = predictions_2d.shape[1]
            if n_choices < pad_to_size:
                # Pad each row using pad_predictions function
                padded_predictions_list = []
                for row in predictions_2d:
                    row_list = row.tolist()
                    padded_row = pad_predictions(row_list, max_num_answers=pad_to_size)
                    padded_predictions_list.append(padded_row)
                predictions_2d = np.array(padded_predictions_list)
                print(f"  Padded predictions from {n_choices} to {pad_to_size} choices")

        # Check consistency of number of questions
        if n_questions is None:
            n_questions = model_n_questions
        elif n_questions != model_n_questions:
            raise ValueError(
                f"Model {model_id} has {model_n_questions} questions, "
                f"but expected {n_questions}"
            )

        # Add to lists
        all_predictions.append(predictions_2d)
        all_correctness.append(correctness_1d)

        # Determine model index (append to existing or use new index)
        if existing_outputs is not None:
            # Find the next available index
            max_existing_idx = max(existing_outputs["Models"].values()) if existing_outputs["Models"] else -1
            model_index = max_existing_idx + 1
        else:
            model_index = model_idx

        models_map[model_id] = model_index
        print(f"  Added model {model_id} at index {model_index}")

    # Stack all models: (n_models, n_questions, n_choices) and (n_models, n_questions, 1)
    if existing_outputs is not None and len(all_predictions) == 0:
        print("No new models to add, keeping existing target_outputs")
        return existing_outputs

    if len(all_predictions) > 0:
        new_predictions = np.stack(all_predictions)  # (n_new_models, n_questions, n_choices)
        new_correctness = np.stack(all_correctness)[:, :, np.newaxis]  # (n_new_models, n_questions, 1)

        if existing_outputs is not None:
            # Concatenate with existing
            predictions = np.concatenate([existing_outputs["predictions"], new_predictions], axis=0)
            correctness = np.concatenate([existing_outputs["correctness"], new_correctness], axis=0)
            # Merge models map
            models_map = {**existing_outputs["Models"], **models_map}
            # Use existing datapoints and scenarios (should be the same)
            datapoints_map = existing_outputs["Datapoints"]
            scenarios_map = existing_outputs["Scenarios"]
        else:
            # Create new
            predictions = new_predictions
            correctness = new_correctness
            # Create Datapoints mapping: idx -> idx
            datapoints_map = {i: i for i in range(n_questions)}

            # Extract scenario base name
            if scenario.startswith("harness_"):
                scenario_base = scenario.replace("harness_", "")
                if "_" in scenario_base:
                    scenario_base = scenario_base.split("_")[0]
            else:
                scenario_base = scenario

            # Create Scenarios mapping: scenario_name -> list of all datapoint indices
            scenarios_map = {scenario_base: list(range(n_questions))}
    else:
        # Only existing models, no new ones
        predictions = existing_outputs["predictions"]
        correctness = existing_outputs["correctness"]
        models_map = existing_outputs["Models"]
        datapoints_map = existing_outputs["Datapoints"]
        scenarios_map = existing_outputs["Scenarios"]

    target_outputs = {
        "predictions": predictions,
        "correctness": correctness,
        "Models": models_map,
        "Datapoints": datapoints_map,
        "Scenarios": scenarios_map,
    }

    # Save to pickle file
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    dump_pickle(target_outputs, output_path)

    print(f"\nSaved target_outputs to {output_path}")
    print(f"Shape of predictions: {target_outputs['predictions'].shape}")
    print(f"Shape of correctness: {target_outputs['correctness'].shape}")
    print(f"Number of models: {len(target_outputs['Models'])}")
    print(f"Number of datapoints: {len(target_outputs['Datapoints'])}")
    print(f"Scenarios: {list(target_outputs['Scenarios'].keys())}")
    print(f"Model IDs: {list(target_outputs['Models'].keys())}")

    return target_outputs



def openllmname_to_hf_model_id(openllm_name):
    assert openllm_name.startswith(
        "open-llm-leaderboard/"
    ), "OpenLLM name must start with 'open-llm-leaderboard/'"
    post_details = openllm_name.split("open-llm-leaderboard/details_")[1]
    creator, model = post_details.split("__")
    hf_model_id = f"{creator}/{model}"
    return hf_model_id


def read_per_model_info(model_id, timestamp, scenario, cache_dir=CACHE_DIR):
    # model_id = "meta-llama/Llama-2-13b-hf"
    creator, model = tuple(model_id.split("/"))
    model_details_id = "open-llm-leaderboard/details_{:}__{:}".format(
        creator, model
    )

    # s = "harness_hendrycksTest_abstract_algebra_5"
    s = scenario
    aux = load_dataset(model_details_id, s, cache_dir=cache_dir)
    print("Available timestamps:")
    print(aux.keys())

    # Attempt to create a table with all available columns.
    latest_data = aux[timestamp]

    # The structure may be a dict with lists as values, or list of dicts.
    # We need to check what we have.

    if isinstance(latest_data, dict):
        # If values are lists of the same length, treat as column-wise.
        lens = [len(v) for v in latest_data.values() if isinstance(v, list)]
        if lens and all(l == lens[0] for l in lens):
            # Looks like column-wise dict of lists.
            df = pd.DataFrame(latest_data)
        else:
            # Dict where each key might be a scalar or something else.
            df = pd.DataFrame([latest_data])
    else:
        # Try to coerce into DataFrame anyway
        df = pd.DataFrame(latest_data)

    # print(df)
    return df


def compute_accuracy(df, metric_key="acc"):
    # Compute accuracy as number of rows where df["metrics"]["acc"] == 1
    # Each element in df["metrics"] is likely a dict with an 'acc' key
    if "metrics" in df.columns:
        num_correct_acc = sum(1 for metric in df["metrics"] if metric.get('acc') == 1)
        num_correct_norm_acc = sum(1 for metric in df["metrics"] if metric.get('norm_acc') == 1)

    else:
        num_correct_acc = sum(1 for acc in df["acc"] if acc == 1)
        num_correct_norm_acc = sum(1 for acc in df["acc_norm"] if acc == 1)

    acc = num_correct_acc / len(df)
    norm_acc = num_correct_norm_acc / len(df)
    print(f"Accuracy (number of acc == 1): {num_correct_acc} / {len(df)}")
    print(f"Accuracy (number of acc == 1): {acc}")
    print(f"Accuracy (number of norm_acc == 1): {num_correct_norm_acc} / {len(df)}")
    print(f"Accuracy (number of norm_acc == 1): {norm_acc}")

    return acc, norm_acc, num_correct_acc, num_correct_norm_acc


def save_prompts(model_dfs, save_path):
    import json

    if not isinstance(model_dfs, list):
        model_dfs = [model_dfs]

    # Extract the relevant columns from model_df_0819
    data_to_save = []
    for model_df_ in model_dfs:
        for idx, row in model_df_.iterrows():
            choices = row.get("choices")
            gold = row.get("gold")
            entry = {
                "example": row.get("example") + " " + choices[int(gold)],
                "full_prompt": row.get("full_prompt"),
                "query": row.get("query"),
                "choices": choices,
                "gold": gold,
            }
            data_to_save.append(entry)

    # Save to json file
    with open(save_path, "w") as outfile:
        json.dump(data_to_save, outfile, indent=2)

    print(f"Saved {len(data_to_save)} records with 'example', 'full_prompt', and 'query' to {save_path}.")

# Analyze

## Save prompts

### Create mmlu_prompts_examples.json

In [3]:
model_dfs = []
for scenario in tqdm(MMLU_SUBSCENARIOS):
    model_df_per_scenario = read_per_model_info(
        model_id="meta-llama/Llama-2-13b-hf",
        # timestamp='2023_08_19T22_35_38.117975',
        timestamp='latest',
        scenario=scenario,
        cache_dir=CACHE_DIR
    )
    model_dfs.append(model_df_per_scenario)

  2%|▏         | 1/57 [00:03<03:17,  3.53s/it]

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])


  4%|▎         | 2/57 [00:04<02:05,  2.28s/it]

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])


  5%|▌         | 3/57 [00:06<01:42,  1.90s/it]

...

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])


 98%|█████████▊| 56/57 [01:35<00:01,  1.69s/it]

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])


100%|██████████| 57/57 [01:37<00:00,  1.71s/it]

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])


In [8]:
save_prompts(model_dfs, "mmlu_prompts_examples.json")

Saved 14042 records with 'example', 'full_prompt', and 'query' to mmlu_prompts_examples.json.


## Compare produced outputs with presaved

In [24]:
model_df_0819 = read_per_model_info(
    model_id="meta-llama/Llama-2-13b-hf",
    # timestamp='2023_08_19T22_35_38.117975',
    timestamp='latest',
    scenario="harness_hellaswag_10",
    cache_dir=CACHE_DIR
)

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])


In [5]:
compute_accuracy(model_df_0819)

Accuracy (number of acc == 1): 6085 / 10042
Accuracy (number of acc == 1): 0.6059549890460068
Accuracy (number of norm_acc == 1): 8131 / 10042
Accuracy (number of norm_acc == 1): 0.809699263095001


(0.6059549890460068, 0.809699263095001)

In [9]:
save_prompts(model_df_0819, "hellaswag_prompts_examples.json")

Saved 10042 records with 'example', 'full_prompt', and 'query' to hellaswag_prompts_examples.json.


In [11]:
model_df_0819.predictions


0        [-99.22494506835938, -106.52214050292969, -225...
1        [-14.67058277130127, -10.640752792358398, -7.1...
2        [-137.53077697753906, -103.27104187011719, -57...
3        [-32.46709060668945, -45.752769470214844, -33....
4        [-139.7080535888672, -62.202911376953125, -116...
                               ...                        
10037    [-88.56155395507812, -87.03571319580078, -95.8...
10038    [-69.46453094482422, -98.91045379638672, -88.5...
10039    [-41.28648376464844, -65.6266860961914, -38.44...
10040    [-25.967365264892578, -21.625568389892578, -37...
10041    [-42.19287872314453, -38.986427307128906, -65....
Name: predictions, Length: 10042, dtype: object

In [21]:
import json



# local_eval_results = load_jsonl("/home/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b.jsonl/meta-llama__Llama-2-13b-hf/samples_hellaswag_prompts_2026-01-05T09-58-06.290694.jsonl")
local_eval_results = load_jsonl("/home/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r58/meta-llama__Llama-2-13b-hf/samples_hellaswag_prompts_2026-01-07T11-42-32.525172.jsonl")

In [18]:
def print_logits(path):
    local_eval_results = load_jsonl(path)
    resps = [r['resps'] for r in local_eval_results]
    new_resps = []
    for r in resps:
        cur_resps = []
        for i in range(len(r)):
            # print(r[i])
            cur_resps.append(float(r[i][0][0]))
        new_resps.append(cur_resps)
        # print(new_resps)
        # break
    new_resps = np.array(new_resps)
    print(new_resps)

In [19]:
print_logits("/home/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b.jsonl/meta-llama__Llama-2-13b-hf/samples_hellaswag_prompts_2026-01-05T09-58-06.290694.jsonl")
print_logits("/home/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r58/meta-llama__Llama-2-13b-hf/samples_hellaswag_prompts_2026-01-07T11-42-32.525172.jsonl")

[[ -97.4375    -106.625     -231.375     -147.125    ]
 [ -15.2265625  -10.4921875   -8.6796875  -30.25     ]
 [-135.75       -98.625      -55.125     -120.75     ]
 ...
 [ -44.1875     -67.875      -40.5        -41.25     ]
 [ -22.796875   -25.859375   -33.90625    -41.375    ]
 [ -38.78125    -38.03125    -60.71875    -61.28125  ]]
[[ -97.875     -107.0625    -227.625     -144.875    ]
 [ -14.3046875  -10.8828125   -7.21875    -31.125    ]
 [-138.125     -102.5        -56.53125   -121.3125   ]
 ...
 [ -41.03125    -67.1875     -38.34375    -40.90625  ]
 [ -24.96875    -21.3125     -37.90625    -45.75     ]
 [ -42.90625    -38.84375    -67.125      -67.875    ]]


In [22]:
local_eval_results

resps = [r['resps'] for r in local_eval_results]

In [23]:
new_resps = []
for r in resps:
    cur_resps = []
    for i in range(len(r)):
        # print(r[i])
        cur_resps.append(float(r[i][0][0]))
    new_resps.append(cur_resps)
    # print(new_resps)
    # break
new_resps = np.array(new_resps)


In [28]:
new_resps

array([[ -97.875    , -107.0625   , -227.625    , -144.875    ],
       [ -14.3046875,  -10.8828125,   -7.21875  ,  -31.125    ],
       [-138.125    , -102.5      ,  -56.53125  , -121.3125   ],
       ...,
       [ -41.03125  ,  -67.1875   ,  -38.34375  ,  -40.90625  ],
       [ -24.96875  ,  -21.3125   ,  -37.90625  ,  -45.75     ],
       [ -42.90625  ,  -38.84375  ,  -67.125    ,  -67.875    ]],
      shape=(10042, 4))

In [25]:
lb_predictions = np.array(model_df_0819.predictions)

In [26]:
lb_predictions_2d = np.stack(lb_predictions)
lb_predictions_2d

array([[ -99.22494507, -106.5221405 , -225.03744507, -142.17416382],
       [ -14.67058277,  -10.64075279,   -7.1043601 ,  -30.81435013],
       [-137.53077698, -103.27104187,  -57.83243179, -119.60269165],
       ...,
       [ -41.28648376,  -65.6266861 ,  -38.44600677,  -42.43055725],
       [ -25.96736526,  -21.62556839,  -37.42005157,  -47.06416702],
       [ -42.19287872,  -38.98642731,  -65.75909424,  -65.87286377]],
      shape=(10042, 4))

In [20]:
print_logits("/home/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r58/meta-llama__Llama-2-13b-hf/samples_hellaswag_prompts_2026-01-07T11-42-32.525172.jsonl")

[[ -97.875     -107.0625    -227.625     -144.875    ]
 [ -14.3046875  -10.8828125   -7.21875    -31.125    ]
 [-138.125     -102.5        -56.53125   -121.3125   ]
 ...
 [ -41.03125    -67.1875     -38.34375    -40.90625  ]
 [ -24.96875    -21.3125     -37.90625    -45.75     ]
 [ -42.90625    -38.84375    -67.125      -67.875    ]]


In [36]:
assert np.allclose(new_resps, lb_predictions_2d, rtol=0.4)

In [37]:
for i in [0, 42, 666]:
    print("Local: {}; Lb: {}".format(new_resps[i], lb_predictions_2d[i]))

Local: [ -97.875  -107.0625 -227.625  -144.875 ]; Lb: [ -99.22494507 -106.5221405  -225.03744507 -142.17416382]
Local: [ -67.9375 -113.375  -121.4375 -133.    ]; Lb: [ -68.8768158  -113.7819519  -118.3768158  -133.52622986]
Local: [-63.875   -44.71875 -77.6875  -73.4375 ]; Lb: [-67.31113434 -44.2445755  -77.51812744 -71.71083832]


## Convert lm-harness data

In [ ]:
paths = """output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__"""

names = """pretrained=abacusai/MetaMath-bagel-34b-v0.2-c1500__COMMA__trust_remote_code=True
pretrained=zhengr/MixTAO-7Bx2-MoE-DPO__COMMA__trust_remote_code=True
pretrained=alignment-handbook/zephyr-7b-sft-full__COMMA__trust_remote_code=True
pretrained=LoSboccacc/orthogonal-2x7B-base__COMMA__trust_remote_code=True
pretrained=rombodawg/Leaderboard-killer-MoE_4x7b__COMMA__trust_remote_code=True
pretrained=rombodawg/Everyone-Coder-4x7b-Base__COMMA__trust_remote_code=True
pretrained=nfaheem/Marcoroni-7b-DPO-Merge__COMMA__trust_remote_code=True
pretrained=deepseek-ai/deepseek-moe-16b-base__COMMA__trust_remote_code=True
pretrained=wang7776/Mistral-7B-Instruct-v0.2-sparsity-20__COMMA__trust_remote_code=True
pretrained=BAAI/Aquila2-34B__COMMA__trust_remote_code=True"""

start_row = 60

model_id_to_result_path_mapping = {}

PROJECT_ROOT = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(""))), "lm-evaluation-harness")

for name, path in zip(names.split("\n"), paths.split("\n")):
    # print(name)
    model_id = name.split("__COMMA__")[0].split("=")[1]
    path = path.replace("__ROW__", str(start_row))
    path = os.path.join(PROJECT_ROOT, path)
    start_row += 1
    # print(path)
    if os.path.exists(path):
        assert len(os.listdir(path)) == 1, f"Path {path} exists but has more than one file"
        folder_name = os.listdir(path)[0]
        path = os.path.join(path, folder_name)
        for file in os.listdir(path):
            if file.endswith(".jsonl"):
                path = os.path.join(path, file)
                break
        assert path.endswith(".jsonl"), f"Path {path} does not end with .jsonl"
        model_id_to_result_path_mapping[model_id] = path
        print(f"Path {path} exists")
    else:
        continue


Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r60/abacusai__MetaMath-bagel-34b-v0.2-c1500/samples_hellaswag_prompts_2026-01-08T10-34-47.120570.jsonl exists
Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r63/LoSboccacc__orthogonal-2x7B-base/samples_hellaswag_prompts_2026-01-08T11-20-27.037399.jsonl exists
Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r65/rombodawg__Everyone-Coder-4x7b-Base/samples_hellaswag_prompts_2026-01-08T09-53-37.527945.jsonl exists
Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r66/nfaheem__Marcoroni-7b-DPO-Merge/samples_hellaswag_prompts_2026-01-08T10-04-25.068130.jsonl exists
Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r68/wang7776__Mistral-7B-Instruct-v0.2-sparsity-20/samples_hellaswag_prompts_2026-01-08T09-39-57.164860.jsonl exists


In [12]:
model_id_to_result_path_mapping

{'abacusai/MetaMath-bagel-34b-v0.2-c1500': '/weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r60/abacusai__MetaMath-bagel-34b-v0.2-c1500/samples_hellaswag_prompts_2026-01-08T10-34-47.120570.jsonl',
 'alignment-handbook/zephyr-7b-sft-full': '/weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r62/alignment-handbook__zephyr-7b-sft-full/samples_hellaswag_prompts_2026-01-08T09-37-48.106657.jsonl',
 'LoSboccacc/orthogonal-2x7B-base': '/weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r63/LoSboccacc__orthogonal-2x7B-base/samples_hellaswag_prompts_2026-01-08T11-20-27.037399.jsonl',
 'rombodawg/Everyone-Coder-4x7b-Base': '/weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r65/rombodawg__Everyone-Coder-4x7b-Base/samples_hellaswag_prompts_2026-01-08T09-53-37.527945.jsonl',
 'nfaheem/Marcoroni-7b-DPO-Merge': '/weka/oh/arubinstein17/github/lm-evaluation-harness/

In [18]:
# Define scenario
scenario = "harness_hellaswag_10"

# Convert and save to target_outputs.pkl
# This will append to existing file if it exists, or create a new one
target_outputs = convert_model_paths_to_target_outputs(
    model_id_to_path_mapping=model_id_to_result_path_mapping,
    scenario=scenario,
    metric="acc_norm",
    output_path="/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_lm_harness_pad_to_size_acc_norm.pkl",
    pad_to_size=31
)

Creating new target_outputs file at /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_lm_harness_pad_to_size_acc_norm.pkl

Processing model 1/6: abacusai/MetaMath-bagel-34b-v0.2-c1500
  Found JSONL file: /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r60/abacusai__MetaMath-bagel-34b-v0.2-c1500/samples_hellaswag_prompts_2026-01-08T10-34-47.120570.jsonl
  Padded predictions from 4 to 31 choices
  Added model abacusai/MetaMath-bagel-34b-v0.2-c1500 at index 0

Processing model 2/6: alignment-handbook/zephyr-7b-sft-full
  Found JSONL file: /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r62/alignment-handbook__zephyr-7b-sft-full/samples_hellaswag_prompts_2026-01-08T09-37-48.106657.jsonl
  Padded predictions from 4 to 31 choices
  Added model alignment-handbook/zephyr-7b-sft-full at index 1

Processing model 3/6: LoSboccacc/orthogonal-2x7B-base
  Found JSONL file: /we

### Debug

In [ ]:
# Example: Convert model_id to path mappings to target_outputs format
# Define your mapping from model_id to local file path
model_id_to_result_path_mapping = {
    "meta-llama/Llama-2-13b-hf": "/home/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b.jsonl/meta-llama__Llama-2-13b-hf/samples_hellaswag_prompts_2026-01-05T09-58-06.290694.jsonl",
    # Add more mappings as needed
    # "another-model/ModelName": "/path/to/another/model/directory",
}

# Define scenario
scenario = "harness_hellaswag_10"

# Convert and save to target_outputs.pkl
# This will append to existing file if it exists, or create a new one
target_outputs = convert_model_paths_to_target_outputs(
    model_id_to_path_mapping=model_id_to_result_path_mapping,
    scenario=scenario,
    output_path="/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_lm_harness.pkl"
)

In [23]:
paths = """output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r79
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__
output/hellaswag_050126/llama13b_r__ROW__"""
# paths = """output/hellaswag_050126/llama13b_r__ROW__
# output/hellaswag_050126/llama13b_r__ROW__
# output/hellaswag_050126/llama13b_r80
# output/hellaswag_050126/llama13b_r__ROW__
# output/hellaswag_050126/llama13b_r__ROW__
# output/hellaswag_050126/llama13b_r__ROW__
# output/hellaswag_050126/llama13b_r__ROW__
# output/hellaswag_050126/llama13b_r__ROW__
# output/hellaswag_050126/llama13b_r__ROW__
# output/hellaswag_050126/llama13b_r__ROW__"""

names = """pretrained=abacusai/MetaMath-bagel-34b-v0.2-c1500__COMMA__trust_remote_code=True
pretrained=zhengr/MixTAO-7Bx2-MoE-DPO__COMMA__trust_remote_code=True
pretrained=alignment-handbook/zephyr-7b-sft-full__COMMA__trust_remote_code=True
pretrained=LoSboccacc/orthogonal-2x7B-base__COMMA__trust_remote_code=True
pretrained=rombodawg/Leaderboard-killer-MoE_4x7b__COMMA__trust_remote_code=True
pretrained=rombodawg/Everyone-Coder-4x7b-Base__COMMA__trust_remote_code=True
pretrained=nfaheem/Marcoroni-7b-DPO-Merge__COMMA__trust_remote_code=True
pretrained=deepseek-ai/deepseek-moe-16b-base__COMMA__trust_remote_code=True
pretrained=wang7776/Mistral-7B-Instruct-v0.2-sparsity-20__COMMA__trust_remote_code=True
pretrained=BAAI/Aquila2-34B__COMMA__trust_remote_code=True"""

start_row = 60

model_id_to_result_path_mapping = {}

PROJECT_ROOT = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(""))), "lm-evaluation-harness")

for name, path in zip(names.split("\n"), paths.split("\n")):
    # print(name)
    model_id = name.split("__COMMA__")[0].split("=")[1]
    path = path.replace("__ROW__", str(start_row))
    path = os.path.join(PROJECT_ROOT, path)
    start_row += 1
    # print(path)
    if os.path.exists(path):
        assert len(os.listdir(path)) == 1, f"Path {path} exists but has more than one file"
        folder_name = os.listdir(path)[0]
        path = os.path.join(path, folder_name)
        for file in os.listdir(path):
            if file.endswith(".jsonl"):
                path = os.path.join(path, file)
                break
        assert path.endswith(".jsonl"), f"Path {path} does not end with .jsonl"
        model_id_to_result_path_mapping[model_id] = path
        print(f"Path {path} exists")
    else:
        continue


Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r60/abacusai__MetaMath-bagel-34b-v0.2-c1500/samples_hellaswag_prompts_2026-01-08T10-34-47.120570.jsonl exists
Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r79/alignment-handbook__zephyr-7b-sft-full/samples_hellaswag_prompts_2026-01-14T14-39-51.006058.jsonl exists
Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r63/LoSboccacc__orthogonal-2x7B-base/samples_hellaswag_prompts_2026-01-08T11-20-27.037399.jsonl exists
Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r65/rombodawg__Everyone-Coder-4x7b-Base/samples_hellaswag_prompts_2026-01-08T09-53-37.527945.jsonl exists
Path /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r66/nfaheem__Marcoroni-7b-DPO-Merge/samples_hellaswag_prompts_2026-01-08T10-04-25.068130.jsonl exists
Path /weka/oh/arubin

In [24]:
# Define scenario
scenario = "harness_hellaswag_10"

# Convert and save to target_outputs.pkl
# This will append to existing file if it exists, or create a new one
target_outputs = convert_model_paths_to_target_outputs(
    model_id_to_path_mapping=model_id_to_result_path_mapping,
    scenario=scenario,
    metric="acc_norm",
    output_path="/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_lm_harness_pad_to_size_acc_norm_debug.pkl",
    pad_to_size=31
)

Creating new target_outputs file at /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_lm_harness_pad_to_size_acc_norm_debug.pkl

Processing model 1/6: abacusai/MetaMath-bagel-34b-v0.2-c1500
  Found JSONL file: /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r60/abacusai__MetaMath-bagel-34b-v0.2-c1500/samples_hellaswag_prompts_2026-01-08T10-34-47.120570.jsonl
  Padded predictions from 4 to 31 choices
  Added model abacusai/MetaMath-bagel-34b-v0.2-c1500 at index 0

Processing model 2/6: alignment-handbook/zephyr-7b-sft-full
  Found JSONL file: /weka/oh/arubinstein17/github/lm-evaluation-harness/output/hellaswag_050126/llama13b_r79/alignment-handbook__zephyr-7b-sft-full/samples_hellaswag_prompts_2026-01-14T14-39-51.006058.jsonl
  Padded predictions from 4 to 31 choices
  Added model alignment-handbook/zephyr-7b-sft-full at index 1

Processing model 3/6: LoSboccacc/orthogonal-2x7B-base
  Found JSONL fil

## Convert lb data to disco-outputs format

In [8]:
list_of_models = [
    'open-llm-leaderboard/details_abacusai__MetaMath-bagel-34b-v0.2-c1500',
    'open-llm-leaderboard/details_zhengr__MixTAO-7Bx2-MoE-DPO',
    'open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full',
    'open-llm-leaderboard/details_LoSboccacc__orthogonal-2x7B-base',
    'open-llm-leaderboard/details_rombodawg__Leaderboard-killer-MoE_4x7b',
    'open-llm-leaderboard/details_rombodawg__Everyone-Coder-4x7b-Base',
    'open-llm-leaderboard/details_nfaheem__Marcoroni-7b-DPO-Merge',
    'open-llm-leaderboard/details_deepseek-ai__deepseek-moe-16b-base',
    'open-llm-leaderboard/details_wang7776__Mistral-7B-Instruct-v0.2-sparsity-20',
    'open-llm-leaderboard/details_BAAI__Aquila2-34B'
]
model_ids = [openllmname_to_hf_model_id(model_name) for model_name in list_of_models]
model_ids = [model_id for model_id in model_ids if model_id in model_id_to_result_path_mapping]

In [16]:
print(model_ids)

['abacusai/MetaMath-bagel-34b-v0.2-c1500', 'alignment-handbook/zephyr-7b-sft-full', 'LoSboccacc/orthogonal-2x7B-base', 'rombodawg/Everyone-Coder-4x7b-Base', 'nfaheem/Marcoroni-7b-DPO-Merge', 'wang7776/Mistral-7B-Instruct-v0.2-sparsity-20']


In [25]:
# Example: Convert list of model_ids to target_outputs format
# Define your list of model IDs
# model_ids = [
#     "meta-llama/Llama-2-13b-hf",
#     # Add more model IDs here as needed
#     # "another-model/ModelName",
# ]

# Define scenario
scenario = "harness_hellaswag_10"

# Convert and save to target_outputs.pkl
# This will append to existing file if it exists, or create a new one
target_outputs = convert_model_ids_to_target_outputs(
    model_ids=model_ids,
    scenario=scenario,
    metric="acc_norm",
    timestamp='latest',
    cache_dir=CACHE_DIR,
    output_path="/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_pad_to_size.pkl",
    pad_to_size=31
)

Creating new target_outputs file at /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_pad_to_size.pkl

Processing model 1/6: abacusai/MetaMath-bagel-34b-v0.2-c1500
Available timestamps:
dict_keys(['2024_01_17T09_47_33.246115', '2024_01_17T09_50_20.465897', 'latest'])
  Padded predictions from 4 to 31 choices
  Added model abacusai/MetaMath-bagel-34b-v0.2-c1500 at index 0

Processing model 2/6: alignment-handbook/zephyr-7b-sft-full
Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
  Padded predictions from 4 to 31 choices
  Added model alignment-handbook/zephyr-7b-sft-full at index 1

Processing model 3/6: LoSboccacc/orthogonal-2x7B-base
Available timestamps:
dict_keys(['2024_01_16T21_21_27.618218', 'latest'])
  Padded predictions from 4 to 31 choices
  Added model LoSboccacc/orthogonal-2x7B-base at index 2

Processing model 4/6: rombodawg/Everyone-Coder-4x7b-Base
Available timestamp

## Compare lb and lm-eval-harness target_outputs

In [37]:
lm_eval_harness_target_outputs = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_lm_harness_pad_to_size_acc_norm.pkl")
lb_target_outputs = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_pad_to_size.pkl")

lm_eval_harness_target_outputs_debug = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_lm_harness_pad_to_size_acc_norm_debug.pkl")
assert np.allclose(lm_eval_harness_target_outputs_debug["predictions"], lm_eval_harness_target_outputs["predictions"])

# lm_eval_harness_target_outputs_80 = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_target_outputs_r80.pkl")
# assert np.allclose(lm_eval_harness_target_outputs_80["predictions"], lm_eval_harness_target_outputs["predictions"])


In [55]:
anchor_points = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/anchor_points_disagreement.pkl")
with open("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r79.pkl", "rb") as f:
    local_predictions_r79 = pickle.load(f) # eval on all
with open("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r80.pkl", "rb") as f:
    local_predictions_r80 = pickle.load(f) # skip non anchor

# predictions_local = lm_eval_harness_target_outputs["predictions"][:, anchor_points, :]
zephyr_idx = 1
predictions_local = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions.pkl")
predictions_local_debug = lm_eval_harness_target_outputs_debug["predictions"][:, anchor_points, :]

print(local_predictions_r79.shape)
print(predictions_local.shape)
print(predictions_local_debug.shape)
assert np.allclose(predictions_local, predictions_local_debug)
assert np.allclose(predictions_local[zephyr_idx], local_predictions_r79)
assert np.allclose(predictions_local_debug[zephyr_idx], local_predictions_r79)
# assert np.allclose(local_predictions_r79, local_predictions_r80)

(1, 100, 31)
(6, 100, 31)
(6, 100, 31)


In [51]:
!mv /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_target_outputs_r79.pkl /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_target_outputs_r79_old.pkl
!mv /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_target_outputs_r80.pkl /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_target_outputs_r80_old.pkl
!mv /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r79.pkl /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r79_old.pkl
!mv /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r80.pkl /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r80_old.pkl

mv: cannot stat '/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_target_outputs_r79.pkl': No such file or directory
mv: cannot stat '/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_target_outputs_r80.pkl': No such file or directory
mv: cannot stat '/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r79.pkl': No such file or directory
mv: cannot stat '/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r80.pkl': No such file or directory


In [5]:
anchor_points = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/anchor_points_disagreement.pkl")
with open("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r79.pkl", "rb") as f:
    local_predictions_r79 = pickle.load(f) # eval on all
with open("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions_r80.pkl", "rb") as f:
    local_predictions_r80 = pickle.load(f) # skip non anchor
predictions_local = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions.pkl")
assert np.allclose(local_predictions_r80, local_predictions_r79)
assert np.allclose(predictions_local[:, anchor_points, :], local_predictions_r80)

AssertionError: 

In [53]:
local_predictions_r79.shape

(1, 100, 31)

In [54]:
print(local_predictions_r80[:5])

[[[ -79.5  -94.   -87.5 ...   -inf   -inf   -inf]
  [ -94.   -78.   -94.  ...   -inf   -inf   -inf]
  [ -97.5 -106.5 -115.  ...   -inf   -inf   -inf]
  ...
  [ -94.  -113.5  -98.  ...   -inf   -inf   -inf]
  [ -94.5  -97.   -88.  ...   -inf   -inf   -inf]
  [ -93.   -95.   -91.  ...   -inf   -inf   -inf]]]


In [45]:
print(lm_eval_harness_target_outputs["predictions"][:, anchor_points, :][:5])
print(local_predictions_r80[:5])

[[[ -98.0625   -102.6875   -107.       ...        -inf        -inf
          -inf]
  [-125.5      -134.5      -110.5      ...        -inf        -inf
          -inf]
  [ -61.59375  -116.5625    -79.3125   ...        -inf        -inf
          -inf]
  ...
  [ -70.5       -64.6875    -79.875    ...        -inf        -inf
          -inf]
  [ -69.125     -89.3125   -101.6875   ...        -inf        -inf
          -inf]
  [ -27.703125  -26.796875  -29.609375 ...        -inf        -inf
          -inf]]

 [[ -98.        -99.       -114.       ...        -inf        -inf
          -inf]
  [-127.       -133.       -108.5      ...        -inf        -inf
          -inf]
  [ -49.25     -110.        -79.       ...        -inf        -inf
          -inf]
  ...
  [ -66.5       -67.5       -75.5      ...        -inf        -inf
          -inf]
  [ -66.5       -93.5       -99.5      ...        -inf        -inf
          -inf]
  [ -30.375     -23.875     -27.25     ...        -inf        -inf
      

In [6]:
# save only predictions
anchor_points = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/anchor_points_disagreement.pkl")

# save only predictions

predictions_local = lm_eval_harness_target_outputs["predictions"][:, anchor_points, :]
with open("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/local_predictions.pkl", "wb") as f:
    pickle.dump(predictions_local, f)

predictions_lb = lb_target_outputs["predictions"][:, anchor_points, :]
with open("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/lb_predictions.pkl", "wb") as f:
    pickle.dump(predictions_lb, f)



In [17]:
# correctness
print(lb_target_outputs["correctness"].shape)
print(lm_eval_harness_target_outputs["correctness"].shape)
print(np.array(lb_target_outputs["correctness"][0]).mean())
print(np.array(lm_eval_harness_target_outputs["correctness"][0]).mean())
print(lb_target_outputs["correctness"][0][:5])
print(lm_eval_harness_target_outputs["correctness"][0][:5])
print(lb_target_outputs["Models"])
print(lm_eval_harness_target_outputs["Models"])
# predictions
print(lb_target_outputs["predictions"].shape)
print(lm_eval_harness_target_outputs["predictions"].shape)
print(lb_target_outputs["predictions"][0][:5])
print(lm_eval_harness_target_outputs["predictions"][0][:5])

(6, 10042, 1)
(6, 10042, 1)
0.8243377813184625
0.8277235610436168
[[1.]
 [1.]
 [1.]
 [1.]
 [1.]]
[[1.]
 [1.]
 [1.]
 [1.]
 [1.]]
{'abacusai/MetaMath-bagel-34b-v0.2-c1500': 0, 'alignment-handbook/zephyr-7b-sft-full': 1, 'LoSboccacc/orthogonal-2x7B-base': 2, 'rombodawg/Everyone-Coder-4x7b-Base': 3, 'nfaheem/Marcoroni-7b-DPO-Merge': 4, 'wang7776/Mistral-7B-Instruct-v0.2-sparsity-20': 5}
{'abacusai/MetaMath-bagel-34b-v0.2-c1500': 0, 'alignment-handbook/zephyr-7b-sft-full': 1, 'LoSboccacc/orthogonal-2x7B-base': 2, 'rombodawg/Everyone-Coder-4x7b-Base': 3, 'nfaheem/Marcoroni-7b-DPO-Merge': 4, 'wang7776/Mistral-7B-Instruct-v0.2-sparsity-20': 5}
(6, 10042, 31)
(6, 10042, 31)
[[-111.67840576 -112.15322113 -225.25460815 -162.75857544          -inf
           -inf          -inf          -inf          -inf          -inf
           -inf          -inf          -inf          -inf          -inf
           -inf          -inf          -inf          -inf          -inf
           -inf          -inf         

### Debug

In [9]:
# Example: Convert list of model_ids to target_outputs format
# Define your list of model IDs
# model_ids = [
#     "meta-llama/Llama-2-13b-hf",
#     # Add more model IDs here as needed
#     # "another-model/ModelName",
# ]

# Define scenario
scenario = "harness_hellaswag_10"

# Convert and save to target_outputs.pkl
# This will append to existing file if it exists, or create a new one
target_outputs = convert_model_ids_to_target_outputs(
    model_ids=model_ids,
    scenario=scenario,
    metric="acc",
    timestamp='latest',
    cache_dir=CACHE_DIR,
    output_path="/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_pad_to_size_acc.pkl",
    pad_to_size=31
)

Creating new target_outputs file at /home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_pad_to_size_acc.pkl

Processing model 1/6: abacusai/MetaMath-bagel-34b-v0.2-c1500
Available timestamps:
dict_keys(['2024_01_17T09_47_33.246115', '2024_01_17T09_50_20.465897', 'latest'])
  Padded predictions from 4 to 31 choices
  Added model abacusai/MetaMath-bagel-34b-v0.2-c1500 at index 0

Processing model 2/6: alignment-handbook/zephyr-7b-sft-full
Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
  Padded predictions from 4 to 31 choices
  Added model alignment-handbook/zephyr-7b-sft-full at index 1

Processing model 3/6: LoSboccacc/orthogonal-2x7B-base
Available timestamps:
dict_keys(['2024_01_16T21_21_27.618218', 'latest'])
  Padded predictions from 4 to 31 choices
  Added model LoSboccacc/orthogonal-2x7B-base at index 2

Processing model 4/6: rombodawg/Everyone-Coder-4x7b-Base
Available times

In [25]:
lm_eval_harness_target_outputs = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_lm_harness_pad_to_size.pkl")
lb_target_outputs = load_pickle("/home/oh/arubinstein17/github/disco-public/data/model_outputs/hellaswag/target_outputs_from_notebook_pad_to_size_acc.pkl")

In [13]:
# correctness
print(lb_target_outputs["correctness"].shape)
print(lm_eval_harness_target_outputs["correctness"].shape)
print(np.array(lb_target_outputs["correctness"][0]).mean())
print(np.array(lm_eval_harness_target_outputs["correctness"][0]).mean())
print(lb_target_outputs["correctness"][0][:5])
print(lm_eval_harness_target_outputs["correctness"][0][:5])
print(lb_target_outputs["Models"])
print(lm_eval_harness_target_outputs["Models"])
# predictions
print(lb_target_outputs["predictions"].shape)
print(lm_eval_harness_target_outputs["predictions"].shape)
print(lb_target_outputs["predictions"][0][:5])
print(lm_eval_harness_target_outputs["predictions"][0][:5])

(6, 10042, 1)
(6, 10042, 1)
0.6275642302330213
0.6320454092810197
[[1.]
 [0.]
 [1.]
 [1.]
 [1.]]
[[1.]
 [0.]
 [1.]
 [1.]
 [1.]]
{'abacusai/MetaMath-bagel-34b-v0.2-c1500': 0, 'alignment-handbook/zephyr-7b-sft-full': 1, 'LoSboccacc/orthogonal-2x7B-base': 2, 'rombodawg/Everyone-Coder-4x7b-Base': 3, 'nfaheem/Marcoroni-7b-DPO-Merge': 4, 'wang7776/Mistral-7B-Instruct-v0.2-sparsity-20': 5}
{'abacusai/MetaMath-bagel-34b-v0.2-c1500': 0, 'alignment-handbook/zephyr-7b-sft-full': 1, 'LoSboccacc/orthogonal-2x7B-base': 2, 'rombodawg/Everyone-Coder-4x7b-Base': 3, 'nfaheem/Marcoroni-7b-DPO-Merge': 4, 'wang7776/Mistral-7B-Instruct-v0.2-sparsity-20': 5}
(6, 10042, 31)
(6, 10042, 31)
[[-111.67840576 -112.15322113 -225.25460815 -162.75857544          -inf
           -inf          -inf          -inf          -inf          -inf
           -inf          -inf          -inf          -inf          -inf
           -inf          -inf          -inf          -inf          -inf
           -inf          -inf         

## Save questions

In [54]:
model_df_0819 = read_per_model_info(
    model_id="meta-llama/Llama-2-13b-hf",
    # timestamp='2023_08_19T22_35_38.117975',
    timestamp='latest',
    scenario="harness_hendrycksTest_anatomy_5",
    cache_dir=CACHE_DIR
)
compute_accuracy(model_df_0819)

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])
Accuracy (number of acc == 1): 69 / 135
Accuracy (number of acc == 1): 0.5111111111111111
Accuracy (number of norm_acc == 1): 69 / 135
Accuracy (number of norm_acc == 1): 0.5111111111111111


(0.5111111111111111, 0.5111111111111111)

In [51]:
model_df_0819

,acc,acc_norm,choices,cont_tokens,example,full_prompt,gold,hashes,input_tokens,instruction,num_asked_few_shots,num_effective_few_shots,padded,predictions,query,truncated
0,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Gastrulation is the process of\nA. mesoderm fo...,The following are multiple choice questions (w...,1,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[133, 133, 133, 133]","[-1.712890625, -1.275390625, -1.087890625, -1....",Gastrulation is the process of\nA. mesoderm fo...,"[0, 0, 0, 0]"
1,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Blood flows from the right ventricle of the he...,The following are multiple choice questions (w...,2,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[144, 144, 144, 144]","[-1.99609375, -1.87109375, -1.43359375, -0.777...",Blood flows from the right ventricle of the he...,"[0, 0, 0, 0]"
2,1.0,1.0,"[A, B, C, D]","[[319], [350], [315], [360]]","In men, specimens for gonococcal cultures are ...",The following are multiple choice questions (w...,2,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[150, 150, 150, 150]","[-2.69140625, -2.41015625, -0.55029296875, -1....","In men, specimens for gonococcal cultures are ...","[0, 0, 0, 0]"
3,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Primary motor cortex activity results in\nA. b...,The following are multiple choice questions (w...,3,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[125, 125, 125, 125]","[-1.431640625, -2.041015625, -0.80712890625, -...",Primary motor cortex activity results in\nA. b...,"[0, 0, 0, 0]"
4,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Saliva contains an enzyme that acts upon which...,The following are multiple choice questions (w...,0,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[158, 158, 158, 158]","[-1.923828125, -0.314208984375, -2.892578125, ...",Saliva contains an enzyme that acts upon which...,"[0, 0, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,1.0,1.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Which of the following best describes the proc...,The following are multiple choice questions (w...,2,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[145, 145, 145, 145]","[-5.9140625, -4.921875, -0.0248870849609375, -...",Which of the following best describes the proc...,"[0, 0, 0, 0]"
131,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",A possible effect of damage to the third crani...,The following are multiple choice questions (w...,1,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[141, 141, 141, 141]","[-0.9423828125, -1.8173828125, -1.5048828125, ...",A possible effect of damage to the third crani...,"[0, 0, 0, 0]"
132,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",The major concentrations of proprioceptive rec...,The following are multiple choice questions (w...,1,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[79, 79, 79, 79]","[-0.94921875, -1.76171875, -1.66796875, -1.402...",The major concentrations of proprioceptive rec...,"[0, 0, 0, 0]"
133,0.0,0.0,"[A, B, C, D]","[[319], [350], [315], [360]]",Which of the following anatomical regions of a...,The following are multiple choice questions (w...,0,"{'cont_tokens': '8994dd2215003fd2', 'example':...","[[1, 450, 1494, 526, 2999, 7348, 5155, 313, 25...",,5,5,"[147, 147, 147, 147]","[-1.40234375, -1.32421875, -1.41796875, -1.464...",Which of the following anatomical regions of a...,"[0, 0, 0, 0]"


In [53]:
import json

# Extract the relevant columns from model_df_0819
data_to_save = []
for idx, row in model_df_0819.iterrows():
    entry = {
        "example": row.get("example"),
        "full_prompt": row.get("full_prompt"),
        "query": row.get("query"),
        "choices": row.get("choices"),
        "gold": row.get("gold"),
    }
    data_to_save.append(entry)

# Save to json file
with open("anatomy_prompts_examples.json", "w") as outfile:
    json.dump(data_to_save, outfile, indent=2)

print(f"Saved {len(data_to_save)} records with 'example', 'full_prompt', and 'query' to anatomy_prompts_examples.json.")


Saved 135 records with 'example', 'full_prompt', and 'query' to anatomy_prompts_examples.json.


## Compare results from outputs and hf

In [28]:
with open("/home/oh/arubinstein17/github/disco-public/cache/target_outputs2.pkl", "rb") as f:
    target_outputs = pickle.load(f)


In [29]:
print(target_outputs.keys())
print(target_outputs["predictions"].shape)
print(target_outputs["correctness"].shape)
print(target_outputs["Models"])



dict_keys(['predictions', 'correctness', 'Models', 'Datapoints', 'Scenarios'])
(40, 14042, 31)
(40, 14042, 1)
{'open-llm-leaderboard/details_abacusai__MetaMath-bagel-34b-v0.2-c1500': 0, 'open-llm-leaderboard/details_zhengr__MixTAO-7Bx2-MoE-DPO': 1, 'open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full': 2, 'open-llm-leaderboard/details_LoSboccacc__orthogonal-2x7B-base': 3, 'open-llm-leaderboard/details_rombodawg__Leaderboard-killer-MoE_4x7b': 4, 'open-llm-leaderboard/details_moreh__MoMo-70B-lora-1.8.6-DPO': 5, 'open-llm-leaderboard/details_FelixChao__ExtremeDolphin-MoE': 6, 'open-llm-leaderboard/details_rombodawg__Everyone-Coder-4x7b-Base': 7, 'open-llm-leaderboard/details_nfaheem__Marcoroni-7b-DPO-Merge': 8, 'open-llm-leaderboard/details_Swisslex__Mixtral-Orca-v0.1': 9, 'open-llm-leaderboard/details_deepseek-ai__deepseek-moe-16b-base': 10, 'open-llm-leaderboard/details_wang7776__Mistral-7B-Instruct-v0.2-sparsity-20': 11, 'open-llm-leaderboard/details_BAAI__Aquila2-34B':

In [48]:
def acc_from_outputs(target_outputs, target_model_name, scenario_name):

    # target_model_name = 'open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full'
    # scenario_name = "harness_hendrycksTest_abstract_algebra_5"
    abstract_algebra_indices = target_outputs["Scenarios"][scenario_name]
    target_model_id = target_outputs["Models"][target_model_name]
    correctness = target_outputs["correctness"][target_model_id][abstract_algebra_indices]
    acc = sum(correctness) / len(correctness)
    print(f"sum: {sum(correctness)}")
    print(f"len: {len(correctness)}")
    print(f"acc: {acc}")
    # print(target_model_name)
    # print(target_model_id)
    # print(acc)
    return acc

In [44]:
target_model_name = "open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full"
# scenario_name = "harness_hendrycksTest_abstract_algebra_5"
timestamp = '2024_01_16T04_06_55.134598'
for scenario_name in target_outputs["Scenarios"].keys():
    acc_hf = acc_from_hf(target_model_name, scenario_name, timestamp)
    acc_outputs = acc_from_outputs(target_outputs, target_model_name, scenario_name)
    assert acc_hf == acc_outputs, f"Scenario {scenario_name} failed: {acc_hf} != {acc_outputs}"


Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 30 / 100
Accuracy (number of acc == 1): 0.3
Accuracy (number of norm_acc == 1): 0 / 100
Accuracy (number of norm_acc == 1): 0.0
Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 77 / 135
Accuracy (number of acc == 1): 0.5703703703703704
Accuracy (number of norm_acc == 1): 0 / 135
Accuracy (number of norm_acc == 1): 0.0


AssertionError: Scenario harness_hendrycksTest_anatomy_5 failed: 0.5703703703703704 != 0.5925925925925926

In [56]:
target_model_name = "open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full"

timestamp = '2024_01_16T04_06_55.134598'
# timestamp = '2024_01_16T04_10_47.293422'
# scenario_name = "harness_hendrycksTest_anatomy_5"
scenario_name = "harness_hendrycksTest_abstract_algebra_5"
# scenario_name = "harness_arc_challenge_25"
acc_hf = acc_from_hf(target_model_name, scenario_name, timestamp)
acc_outputs = acc_from_outputs(target_outputs, target_model_name, scenario_name)
assert acc_hf == acc_outputs, f"Scenario {scenario_name} failed: {acc_hf} != {acc_outputs}"

Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 30 / 100
Accuracy (number of acc == 1): 0.3
Accuracy (number of norm_acc == 1): 0 / 100
Accuracy (number of norm_acc == 1): 0.0
sum: [30.]
len: 100
acc: [0.3]


In [6]:
target_model_name = "open-llm-leaderboard/details_alignment-handbook__zephyr-7b-sft-full"
total_len = 0
total_num_correct_acc = 0
total_num_correct_norm_acc = 0
timestamp = 'latest'
for scenario_name in tqdm(MMLU_SUBSCENARIOS):
    acc, norm_acc, num_correct_acc, num_correct_norm_acc, len_df = acc_from_hf(target_model_name, scenario_name, timestamp)
    print(f"Scenario {scenario_name} acc: {acc}, norm_acc: {norm_acc}, num_correct_acc: {num_correct_acc}, num_correct_norm_acc: {num_correct_norm_acc}, len_df: {len_df}")
    print(f"Scenario {scenario_name} acc: {acc}, norm_acc: {norm_acc}, num_correct_acc: {num_correct_acc}, num_correct_norm_acc: {num_correct_norm_acc}, len_df: {len_df}")
    print(f"Scenario {scenario_name} acc: {acc}, norm_acc: {norm_acc}, num_correct_acc: {num_correct_acc}, num_correct_norm_acc: {num_correct_norm_acc}, len_df: {len_df}")
    print(f"Scenario {scenario_name} acc: {acc}, norm_acc: {norm_acc}, num_correct_acc: {num_correct_acc}, num_correct_norm_acc: {num_correct_norm_acc}, len_df: {len_df}")
    print(f"Scenario {scenario_name} acc: {acc}, norm_acc: {norm_acc}, num_correct_acc: {num_correct_acc}, num_correct_norm_acc: {num_correct_norm_acc}, len_df: {len_df}")
    total_len += len_df
    total_num_correct_acc += num_correct_acc
    total_num_correct_norm_acc += num_correct_norm_acc
print(f"Total len: {total_len}")
print(f"Total num_correct_acc: {total_num_correct_acc}")
print(f"Total num_correct_norm_acc: {total_num_correct_norm_acc}")
print(f"Total acc: {total_num_correct_acc / total_len}")
print(f"Total norm_acc: {total_num_correct_norm_acc / total_len}")




  2%|▏         | 1/57 [00:05<04:50,  5.19s/it]

Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 29 / 100
Accuracy (number of acc == 1): 0.29
Accuracy (number of norm_acc == 1): 0 / 100
Accuracy (number of norm_acc == 1): 0.0
Scenario harness_hendrycksTest_abstract_algebra_5 acc: 0.29, norm_acc: 0.0, num_correct_acc: 29, num_correct_norm_acc: 0, len_df: 100
Scenario harness_hendrycksTest_abstract_algebra_5 acc: 0.29, norm_acc: 0.0, num_correct_acc: 29, num_correct_norm_acc: 0, len_df: 100
Scenario harness_hendrycksTest_abstract_algebra_5 acc: 0.29, norm_acc: 0.0, num_correct_acc: 29, num_correct_norm_acc: 0, len_df: 100
Scenario harness_hendrycksTest_abstract_algebra_5 acc: 0.29, norm_acc: 0.0, num_correct_acc: 29, num_correct_norm_acc: 0, len_df: 100
Scenario harness_hendrycksTest_abstract_algebra_5 acc: 0.29, norm_acc: 0.0, num_correct_acc: 29, num_correct_norm_acc: 0, len_df: 100


  4%|▎         | 2/57 [00:06<02:52,  3.13s/it]

Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 78 / 135
Accuracy (number of acc == 1): 0.5777777777777777
Accuracy (number of norm_acc == 1): 0 / 135
Accuracy (number of norm_acc == 1): 0.0
Scenario harness_hendrycksTest_anatomy_5 acc: 0.5777777777777777, norm_acc: 0.0, num_correct_acc: 78, num_correct_norm_acc: 0, len_df: 135
Scenario harness_hendrycksTest_anatomy_5 acc: 0.5777777777777777, norm_acc: 0.0, num_correct_acc: 78, num_correct_norm_acc: 0, len_df: 135
Scenario harness_hendrycksTest_anatomy_5 acc: 0.5777777777777777, norm_acc: 0.0, num_correct_acc: 78, num_correct_norm_acc: 0, len_df: 135
Scenario harness_hendrycksTest_anatomy_5 acc: 0.5777777777777777, norm_acc: 0.0, num_correct_acc: 78, num_correct_norm_acc: 0, len_df: 135
Scenario harness_hendrycksTest_anatomy_5 acc: 0.5777777777777777, norm_acc: 0.0, num_correct_acc: 78, num_correct_norm_acc: 0, len_df: 135


Generating 2024_01_16T04_06_55.134598 split: 152 examples [00:00, 11676.88 examples/s]
Generating 2024_01_16T04_10_47.293422 split: 152 examples [00:00, 15648.85 examples/s]
Generating latest split: 152 examples [00:00, 16105.86 examples/s]
  5%|▌         | 3/57 [00:08<02:09,  2.39s/it]

...

Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 80 / 100
Accuracy (number of acc == 1): 0.8
Accuracy (number of norm_acc == 1): 0 / 100
Accuracy (number of norm_acc == 1): 0.0
Scenario harness_hendrycksTest_us_foreign_policy_5 acc: 0.8, norm_acc: 0.0, num_correct_acc: 80, num_correct_norm_acc: 0, len_df: 100
Scenario harness_hendrycksTest_us_foreign_policy_5 acc: 0.8, norm_acc: 0.0, num_correct_acc: 80, num_correct_norm_acc: 0, len_df: 100
Scenario harness_hendrycksTest_us_foreign_policy_5 acc: 0.8, norm_acc: 0.0, num_correct_acc: 80, num_correct_norm_acc: 0, len_df: 100
Scenario harness_hendrycksTest_us_foreign_policy_5 acc: 0.8, norm_acc: 0.0, num_correct_acc: 80, num_correct_norm_acc: 0, len_df: 100
Scenario harness_hendrycksTest_us_foreign_policy_5 acc: 0.8, norm_acc: 0.0, num_correct_acc: 80, num_correct_norm_acc: 0, len_df: 100


Generating 2024_01_16T04_06_55.134598 split: 166 examples [00:00, 12935.76 examples/s]
Generating 2024_01_16T04_10_47.293422 split: 166 examples [00:00, 22826.52 examples/s]
Generating latest split: 166 examples [00:00, 21222.74 examples/s]
 98%|█████████▊| 56/57 [01:41<00:01,  1.77s/it]

Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 84 / 166
Accuracy (number of acc == 1): 0.5060240963855421
Accuracy (number of norm_acc == 1): 0 / 166
Accuracy (number of norm_acc == 1): 0.0
Scenario harness_hendrycksTest_virology_5 acc: 0.5060240963855421, norm_acc: 0.0, num_correct_acc: 84, num_correct_norm_acc: 0, len_df: 166
Scenario harness_hendrycksTest_virology_5 acc: 0.5060240963855421, norm_acc: 0.0, num_correct_acc: 84, num_correct_norm_acc: 0, len_df: 166
Scenario harness_hendrycksTest_virology_5 acc: 0.5060240963855421, norm_acc: 0.0, num_correct_acc: 84, num_correct_norm_acc: 0, len_df: 166
Scenario harness_hendrycksTest_virology_5 acc: 0.5060240963855421, norm_acc: 0.0, num_correct_acc: 84, num_correct_norm_acc: 0, len_df: 166
Scenario harness_hendrycksTest_virology_5 acc: 0.5060240963855421, norm_acc: 0.0, num_correct_acc: 84, num_correct_norm_acc: 0, len_df: 166


Generating 2024_01_16T04_06_55.134598 split: 171 examples [00:00, 20111.21 examples/s]
Generating 2024_01_16T04_10_47.293422 split: 171 examples [00:00, 20032.57 examples/s]
Generating latest split: 171 examples [00:00, 22930.69 examples/s]
100%|██████████| 57/57 [01:42<00:00,  1.80s/it]

Available timestamps:
dict_keys(['2024_01_16T04_06_55.134598', '2024_01_16T04_10_47.293422', 'latest'])
Accuracy (number of acc == 1): 140 / 171
Accuracy (number of acc == 1): 0.8187134502923976
Accuracy (number of norm_acc == 1): 0 / 171
Accuracy (number of norm_acc == 1): 0.0
Scenario harness_hendrycksTest_world_religions_5 acc: 0.8187134502923976, norm_acc: 0.0, num_correct_acc: 140, num_correct_norm_acc: 0, len_df: 171
Scenario harness_hendrycksTest_world_religions_5 acc: 0.8187134502923976, norm_acc: 0.0, num_correct_acc: 140, num_correct_norm_acc: 0, len_df: 171
Scenario harness_hendrycksTest_world_religions_5 acc: 0.8187134502923976, norm_acc: 0.0, num_correct_acc: 140, num_correct_norm_acc: 0, len_df: 171
Scenario harness_hendrycksTest_world_religions_5 acc: 0.8187134502923976, norm_acc: 0.0, num_correct_acc: 140, num_correct_norm_acc: 0, len_df: 171
Scenario harness_hendrycksTest_world_religions_5 acc: 0.8187134502923976, norm_acc: 0.0, num_correct_acc: 140, num_correct_norm_a